# 4장 2강: 교차검증 전략과 데이터 누수 방지

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 불균형 분류 문제에서 적절한 교차검증 방법을 선택하고, 데이터 누수를 방지하는 방법을 연습합니다.
이번 실습은 거래 간 시간 의존성을 고려하지 않는 분류 연습으로 StratifiedKFold를 사용하며, 실제 미래 거래 예측을 평가할 때는 시간 순서를 보존해 분할한다.

- KFold 교차검증의 기본 원리를 확인합니다.
- 불균형 분류에서 StratifiedKFold가 필요한 이유를 확인합니다.
- KFold / StratifiedKFold / TimeSeriesSplit의 선택 기준을 구분합니다.
- 잘못된 전처리 순서가 데이터 누수를 만들 수 있음을 이해합니다.
- `Pipeline`을 이용해 전처리와 모델 학습을 교차검증 안에서 안전하게 연결합니다.
- SMOTE를 사용할 때 `imbalanced-learn Pipeline`을 이용해 Train Fold에만 적용되도록 구성합니다.

> 이 실습에서는 4장 2강 교안에 나온 교차검증, 데이터 누수, Pipeline 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습은 불균형 분류 문제이므로 `Class` 비율을 유지하는 교차검증 전략이 중요합니다.

## 실습 준비

1. 데이터를 불러옵니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 입력 특성 X로 사용합니다.
4. `Class`를 정답 y로 사용합니다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

data = pd.read_csv("creditcard.csv")

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(y.value_counts().to_frame("count"))

print("\n[Class 비율]")
display(y.value_counts(normalize=True).to_frame("ratio"))

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727


---

# 필수 1. KFold와 StratifiedKFold 비교

## 배경

KFold는 데이터를 여러 Fold로 나누어 반복 평가합니다.

하지만 Credit Card Fraud처럼 클래스 비율이 크게 불균형한 데이터에서는 Fold마다 사기 거래 비율이 달라질 수 있습니다.

StratifiedKFold는 각 Fold에서 원본 데이터와 비슷한 클래스 비율을 유지하도록 분할합니다.

## 문제 1. KFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `KFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Fold의 Validation 데이터에서 `Class=1` 비율을 계산합니다.
3. Fold별 사기 거래 비율을 출력합니다.

### 확인 포인트

- KFold는 클래스 비율 자체를 유지하도록 설계된 방법이 아님을 이해했는가?
- Fold별 소수 클래스 비율이 조금씩 달라질 수 있음을 확인했는가?

In [2]:
# TODO
# KFold의 각 Validation Fold에서 Class=1 비율을 확인하세요.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

kf_rows = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):
    y_val = y.iloc[val_idx]
    kf_rows.append({
        "fold": fold,
        "val_size": len(y_val),
        "fraud_count": int(y_val.sum()),
        "fraud_ratio": y_val.mean()
    })

kf_fold_df = pd.DataFrame(kf_rows)

print("[KFold] Fold별 Validation 사기 거래 비율")
display(kf_fold_df)

print("\n전체 사기 거래 비율:", y.mean())
print("KFold 비율 표준편차:", kf_fold_df["fraud_ratio"].std())

[KFold] Fold별 Validation 사기 거래 비율


,fold,val_size,fraud_count,fraud_ratio
0,1,56962,98,0.001720
1,2,56962,93,0.001633
2,3,56961,95,0.001668
3,4,56961,101,0.001773
4,5,56961,105,0.001843



전체 사기 거래 비율: 0.001727485630620034
KFold 비율 표준편차: 8.383687515850591e-05


#### 전체 28만 4천건을 다섯 묶음으로 나누면 두 검정 묶음은 56,962, 56,961건 정도로 나뉨
#### 앞서 Fraud_ratio의 비율을 확인하여 각 폴드에 들어간 사기 건수는 98, 93, 95, 101, 105건, 합은 492건

#### KFold의 묶음은 전체 크기를 비슷하게 맞추려고 함. 단 클래스별 개수를 맞추도록 설계된 방법은 아님
#### 이번에는 표본이 어느 정도 크기 때문에 유사한 크기로 사기들이 각 폴드에 포함되지만
#### KFold의 각 폴드들에 데이터셋에 따라 양성이 포함되지 않을 수도 있다.

## 문제 2. StratifiedKFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Validation Fold의 `Class=1` 비율을 계산합니다.
3. 원본 전체 데이터의 사기 거래 비율과 비교합니다.

### 확인 포인트

- 각 Fold의 클래스 비율이 원본과 비슷하게 유지되는가?
- 불균형 분류에서 StratifiedKFold가 적합한 이유를 설명할 수 있는가?

In [3]:
# TODO
# StratifiedKFold의 각 Validation Fold에서 Class=1 비율을 확인하세요.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

skf_rows = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    y_val = y.iloc[val_idx]
    skf_rows.append({
        "fold": fold,
        "val_size": len(y_val),
        "fraud_count": int(y_val.sum()),
        "fraud_ratio": y_val.mean()
    })

skf_fold_df = pd.DataFrame(skf_rows)

print("[StratifiedKFold] Fold별 Validation 사기 거래 비율")
display(skf_fold_df)

print("\n전체 사기 거래 비율:", y.mean())
print("StratifiedKFold 비율 표준편차:", skf_fold_df["fraud_ratio"].std())

# 두 방법의 Fold별 비율을 나란히 비교
compare_ratio = pd.DataFrame({
    "fold": kf_fold_df["fold"],
    "KFold_ratio": kf_fold_df["fraud_ratio"],
    "StratifiedKFold_ratio": skf_fold_df["fraud_ratio"]
})

print("\n[Fold별 사기 거래 비율 비교]")
display(compare_ratio)

[StratifiedKFold] Fold별 Validation 사기 거래 비율


,fold,val_size,fraud_count,fraud_ratio
0,1,56962,99,0.001738
1,2,56962,99,0.001738
2,3,56961,98,0.001720
3,4,56961,98,0.001720
4,5,56961,98,0.001720



전체 사기 거래 비율: 0.001727485630620034
StratifiedKFold 비율 표준편차: 9.599034695267571e-06

[Fold별 사기 거래 비율 비교]


,fold,KFold_ratio,StratifiedKFold_ratio
0,1,0.001720,0.001738
1,2,0.001633,0.001738
2,3,0.001668,0.001720
3,4,0.001773,0.001720
4,5,0.001843,0.001720


#### 전체 비율을 봤을떄 0.0017, 약 0.17%정도
#### 사기 492건을 5로 나누면 98.4건, 0.4건을 쪼갤 수 없기 때문에 99건 포함하여 비교적 고르게 각 폴드에 사기 건수가 분포
#### 층화(각 폴드의 클래스 비율을 균등하게 맞추는 것), 클래스 구성 차이를 줄이는 데 도움이 되지만
#### 거리의 특성 분포까지 동일하게 맞출 수는 없기 떄문에 각 폴드의 결과가 같지는 않다.

## 문제 3. 두 교차검증 방법으로 F1-score를 비교하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 사용합니다.
2. KFold로 5-Fold F1-score를 계산합니다.
3. StratifiedKFold로 5-Fold F1-score를 계산합니다.
4. 각 방법의 Fold별 점수, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 한 번의 점수가 아니라 여러 Fold의 평균과 편차를 확인했는가?
- Credit Card Fraud와 같은 불균형 분류에서는 어떤 방법이 더 안전한 기본 선택인지 설명할 수 있는가?

In [4]:
# TODO
# KFold와 StratifiedKFold의 F1-score 평균과 표준편차를 비교하세요.

model = LogisticRegression(max_iter=1000)

kf_scores = cross_val_score(model, X, y, cv=kf, scoring="f1")
skf_scores = cross_val_score(model, X, y, cv=skf, scoring="f1")

print("[KFold] Fold별 F1-score:", np.round(kf_scores, 4))
print("[KFold] 평균:", kf_scores.mean(), "/ 표준편차:", kf_scores.std())

print("\n[StratifiedKFold] Fold별 F1-score:", np.round(skf_scores, 4))
print("[StratifiedKFold] 평균:", skf_scores.mean(), "/ 표준편차:", skf_scores.std())

compare_f1 = pd.DataFrame({
    "cv": ["KFold", "StratifiedKFold"],
    "mean_f1": [kf_scores.mean(), skf_scores.mean()],
    "std_f1": [kf_scores.std(), skf_scores.std()]
})

print("\n[교차검증 방법별 F1-score 비교]")
display(compare_f1)

[KFold] Fold별 F1-score: [0.6829 0.7453 0.716  0.7108 0.7513]
[KFold] 평균: 0.7212967903415802 / 표준편차: 0.024854677807448683

[StratifiedKFold] Fold별 F1-score: [0.6909 0.7168 0.7073 0.7636 0.7614]
[StratifiedKFold] 평균: 0.7279978339720338 / 표준편차: 0.02936961579482164

[교차검증 방법별 F1-score 비교]


,cv,mean_f1,std_f1
0,KFold,0.721297,0.024855
1,StratifiedKFold,0.727998,0.029370


#### Fold별 값의 의미
#### KFold의 0.6829는 첫번째 묶음에서 얻은 F1-Score
#### 전체 거래의 60%를 맞췄다는 의미가 아님

#### 두 가지 결과를 비교하여 결론
#### F1-Score가 0.72로 비교적 비슷하고 StratifiedKFold가 비교적 평균이 약간 높았다.
#### 표준편차는 StratifiedKFold가 더 컸다.
#### 두 가지 결과를 비교했을 때 StratifiedKFold 모델을 선택하는 근거는
#### 극소수인 사기를 검증 묶음에 고르게 분할했기 때문이다.



### Q1. Credit Card Fraud 데이터에서는 KFold와 StratifiedKFold 중 어떤 방법을 우선 사용하는 것이 적절한가요?

**답변:**  
#### StratifiedKFold가 적절함
#### Credit Card 데이터는 소수 클래스인 사기 거래 비율이 매우 낮은 불균형 데이터이기 때문

### Q2. 다음 상황에서 어떤 교차검증 방법을 선택할지 작성하세요.

1. 일반적인 회귀 문제 -> KFold
2. 불균형 분류 문제 -> StratifiedKFold
3. 시간 순서가 중요한 시계열 문제 -> TimeSeriesSplit

#### 교차검증은 무조건 하나의 방법을 고집하여 쓰는 것은 아니고 데이터의 특성에 맞게 선택한다.

---

# 필수 2. Pipeline으로 데이터 누수 방지

## 배경

전처리를 전체 데이터에 먼저 `fit()`한 뒤 교차검증하면 Validation Fold의 정보가 전처리 기준에 포함될 수 있습니다.

교안에서는 이를 방지하기 위해 **전처리와 모델을 Pipeline으로 묶는 방법**을 사용합니다.

## 문제 1. StandardScaler와 Logistic Regression을 Pipeline으로 연결하세요.

### 요구사항

1. `StandardScaler()`와 `LogisticRegression(class_weight="balanced", max_iter=1000)`을 하나의 `Pipeline`으로 만듭니다.
2. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 사용합니다.
3. `cross_val_score()`로 F1-score를 계산합니다.
4. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 각 Fold의 Train 데이터에서만 scaler가 fit되는 구조인가?
- Validation Fold에는 Train에서 학습한 기준으로 transform만 적용되는가?
- 불균형 분류이므로 StratifiedKFold를 사용했는가?

In [5]:
# TODO
# StandardScaler + LogisticRegression을 Pipeline으로 연결하고
# StratifiedKFold로 F1-score를 계산하세요.
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1")

print("[Pipeline + StratifiedKFold] Fold별 F1-score:", np.round(pipe_scores, 4))
print("평균     :", pipe_scores.mean())
print("표준편차 :", pipe_scores.std())

pipe_result = pd.DataFrame({
    "fold": range(1, len(pipe_scores) + 1),
    "f1_score": pipe_scores
})

display(pipe_result)

[Pipeline + StratifiedKFold] Fold별 F1-score: [0.1282 0.12   0.1121 0.1118 0.1175]
평균     : 0.11790859991473175
표준편차 : 0.006026403089505455


,fold,f1_score
0,1,0.128186
1,2,0.120000
2,3,0.112069
3,4,0.111801
4,5,0.117486


#### 결과표의 의미
#### 0.1118~0.1281까지 분포, F1-Score가 전반적으로 높지 않음

#### 앞의 F1-Score는 0.72였는데 지금은 왜 0.12?
#### -> 가장 큰 차이점 : 표준화와 클래스 가중치 적용 여부
#### 클래스 가중치는 사기 탐지를 더 적극적으로 확인하려고 함. -> 정상을 오탐으로 예측할 확률이 증가함.

#### 전체 데이터로 Scaler를 먼저 fit하면 왜 누수라고 부르는지?
#### Validation은 학습 과정에서 직접 풀어보지 않은 자료의 역할을 담당한다.
#### 전체 X로 먼저 Fit하면, Validation의 입력값도 평균과 표준편차에 들어가기 때문에
#### 미리 정답지를 알고 사용한 경우라고 판단하여 데이터 누수라고 한다,

#### 잘못된 순서 : 전체 데이터라 전처리 및 스케일링 -> 변환된 데이터를 cv로 분할 -> 평가

### Q3. Pipeline을 사용하면 스케일링 누수를 방지할 수 있는 이유는 무엇인가요?

**답변:**  
#### 교차 검증의 각 Fold마다 전처리의 Fit()이 Train Fold에만 수행되기 때문

## 문제 2. SMOTE를 교차검증 안에서 안전하게 적용하세요.

### 배경

SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 Validation Fold와 매우 비슷한 합성 데이터가 Train에 포함될 수 있습니다.

교안에서는 `imbalanced-learn`의 Pipeline 안에 SMOTE를 넣어 각 Fold의 Train 부분에만 적용하도록 합니다.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. `LogisticRegression(max_iter=1000)`과 함께 `ImbPipeline`으로 연결합니다.
3. `StratifiedKFold`를 사용합니다.
4. `cross_val_score()`로 F1-score를 계산합니다.
5. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- SMOTE가 교차검증 밖에서 전체 데이터에 적용되지 않았는가?
- 각 Fold의 Train 부분에만 SMOTE가 적용되는 구조인가?

In [6]:
# TODO
# SMOTE + LogisticRegression을 imbalanced-learn Pipeline으로 연결하세요.

smote_pipe = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

smote_scores = cross_val_score(smote_pipe, X, y, cv=cv, scoring="f1")

print("[SMOTE Pipeline + StratifiedKFold] Fold별 F1-score:", np.round(smote_scores, 4))
print("평균     :", smote_scores.mean())
print("표준편차 :", smote_scores.std())

print("\nPipeline(가중치) 평균 - SMOTE 평균 =", pipe_scores.mean() - smote_scores.mean())

[SMOTE Pipeline + StratifiedKFold] Fold별 F1-score: [0.1226 0.1107 0.1065 0.1044 0.1069]
평균     : 0.11022204299107201
표준편차 : 0.006510594105889241

Pipeline(가중치) 평균 - SMOTE 평균 = 0.007686556923659743


#### 이번 SMOTE의 평균 F1은 0.11정도 앞서 진행한 가중치 + 표준화 보다 약 0.007낮음
#### 각 Fold에서 어떤 데이터들이 생성되는지?
#### 정상건수 28만 4천건 정도, 사기 건수 492건, 소수의 클래스를 SMOTE를 적용하여 학습 표본을 크게 높임
#### 전체 데이터셋이 미리 SMOTE를 적용하면 생기는 문제
#### 첫번째, Validation에 들어갈 실제 사기 표본이 합성 표본의 재료나 이웃으로 사용될 수 있다.
#### -> Train에서 이미 학습한 내용들을 Validation 에서 사용될 수 있기 때문에 누수가 발생할 가능성이 있다.
#### 두번쨰, 합성한 전체 데이터를 분할하면 Validation의 사기 비율도 인위적으로 높아질 수 있음.
#### -> 원래 사기 건수가 매우 드물기 때문에 Precision과 F1-Score를 해석하는 조건도 달라짐.


### Q4. SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 왜 문제가 될 수 있나요?

**답변:**  
#### SMOTE가 전체 데이터의 소수 클래스 샘플을 이용해 합성 데이터를 만들면, 이후 Validation Fold와 매우 비슷한 샘플이 Train에 포함될 수가 있다.

---

# 과제 1. 데이터 특성에 맞는 CV와 누수 없는 Pipeline 구성

## 배경

필수 실습에서는 불균형 분류에 적합한 StratifiedKFold와 Pipeline을 직접 사용했습니다.

이번 과제에서는 같은 Credit Card Fraud 데이터에서 **누수 없는 평가 흐름을 스스로 구성하고 설명**합니다.

> 과제는 필수 실습에서 사용한 구성 요소를 다시 연결하는 수준입니다.

## 요구사항

1. `StandardScaler()`를 사용합니다.
2. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
3. 두 단계를 `Pipeline`으로 연결합니다.
4. Credit Card Fraud 데이터 특성에 맞는 교차검증 방법을 직접 선택합니다.
5. `n_splits=5`를 사용합니다.
6. `cross_val_score(..., scoring="f1")`로 성능을 평가합니다.
7. Fold별 F1-score, 평균, 표준편차를 출력합니다.
8. 왜 해당 교차검증 방법을 선택했는지 설명합니다.
9. 다음 잘못된 순서를 데이터 누수 관점에서 설명합니다.

```text
전체 X로 StandardScaler.fit_transform()
        ↓
교차검증 수행
```

10. 시계열 데이터라면 같은 방법을 그대로 사용하면 안 되는 이유와 어떤 CV를 선택해야 하는지 작성합니다.

11. `build_pipeline(model)` 함수를 작성하여 `StandardScaler`와 전달받은 모델을 연결한 Pipeline을 반환합니다. 이 함수 안에서는 데이터를 학습하거나 변환하지 않습니다.
12. `evaluate_pipeline(X, y, model, cv, scoring="f1")` 함수를 작성합니다. 11번 함수를 이용해 Pipeline을 만들고 교차검증한 뒤, Fold별 점수·평균·표준편차를 반환합니다. 모델·CV·평가지표를 함수 인자로 교체할 수 있어야 합니다.
13. 12번 함수를 실제 호출하여 요구사항 7번 결과를 출력합니다. CV에서 데이터를 섞는 경우 `shuffle=True, random_state=42`를 지정하고, 모델 등 무작위성이 있는 구성 요소도 가능한 경우 `random_state=42`로 고정합니다.
14. 함수의 재사용 방법과 재현성 설정을 설명하고 Q5~Q8에 답합니다. 별도 `.py` 모듈 제작이나 다른 모델의 추가 학습은 요구하지 않습니다.

### 확인 포인트

- 불균형 분류 특성에 맞는 CV를 선택했는가?
- Pipeline이 전처리 누수를 방지하는 구조임을 설명했는가?
- 평균뿐 아니라 Fold별 점수와 표준편차도 확인했는가?
- TimeSeriesSplit의 사용 상황을 구분할 수 있는가?
- 함수 인자로 모델·CV·평가지표를 바꿀 수 있는가?
- 함수 호출로 전처리·학습·평가가 연결되고 점수가 반환되는가?
- 재현성을 위한 난수 설정을 명시했는가?


In [7]:
# TODO
# build_pipeline(model)과 evaluate_pipeline(X, y, model, cv, scoring)를 작성하세요.
# 적절한 5-Fold CV와 모델을 전달하고 F1 점수·평균·표준편차를 출력하세요.
# 과제 1. 데이터 특성에 맞는 CV와 누수 없는 Pipeline 구성

def build_pipeline(model):
    """StandardScaler와 전달받은 model을 Pipeline으로 연결해 반환합니다.
    이 함수 안에서는 데이터를 학습하거나 변환하지 않습니다."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])


def evaluate_pipeline(X, y, model, cv, scoring="f1"):
    """build_pipeline으로 Pipeline을 만들어 교차검증하고
    Fold별 점수, 평균, 표준편차를 반환합니다."""
    pipeline = build_pipeline(model)
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring=scoring)
    return scores, scores.mean(), scores.std()


task_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
task_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

task_scores, task_mean, task_std = evaluate_pipeline(X, y, task_model, task_cv, scoring="f1")

print("선택한 CV: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)")
print("Fold별 F1-score:", np.round(task_scores, 4))
print("평균 F1-score  :", task_mean)
print("F1 표준편차    :", task_std)

task_result = pd.DataFrame({
    "fold": range(1, len(task_scores) + 1),
    "f1_score": task_scores
})

display(task_result)

# 함수 재사용 확인 1 - 평가지표만 교체
rec_scores, rec_mean, rec_std = evaluate_pipeline(X, y, task_model, task_cv, scoring="recall")
pre_scores, pre_mean, pre_std = evaluate_pipeline(X, y, task_model, task_cv, scoring="precision")

print("\n[scoring만 교체]")
print("recall    평균:", rec_mean)
print("precision 평균:", pre_mean)

# 함수 재사용 확인 2 - CV만 교체
tscv = TimeSeriesSplit(n_splits=5)
ts_scores, ts_mean, ts_std = evaluate_pipeline(X, y, task_model, tscv, scoring="f1")

print("\n[cv만 TimeSeriesSplit으로 교체]")
print("Fold별 F1:", np.round(ts_scores, 4))
print("평균:", ts_mean, "/ 표준편차:", ts_std)

선택한 CV: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Fold별 F1-score: [0.1282 0.12   0.1121 0.1118 0.1175]
평균 F1-score  : 0.11790859991473175
F1 표준편차    : 0.006026403089505455


,fold,f1_score
0,1,0.128186
1,2,0.120000
2,3,0.112069
3,4,0.111801
4,5,0.117486



[scoring만 교체]
recall    평균: 0.9105545248402391
precision 평균: 0.063056377826583

[cv만 TimeSeriesSplit으로 교체]
Fold별 F1: [0.1906 0.098  0.0867 0.0853 0.0844]
평균: 0.10899165613165071 / 표준편차: 0.04111811333118079


In [ ]:
# def build_pipeline(model):
#     return Pipeline([
#         ("scaler", StandardScaler()),
#         ("model", model)                    # ← ① 받은 model을 그대로 넣음
#     ])


# def evaluate_pipeline(X, y, model, cv, scoring="f1"):
#     pipeline = build_pipeline(model)
#     scores = cross_val_score(
#         pipeline, X, y,
#         cv=cv,                              # ← ② 받은 cv를 그대로 넘김
#         scoring=scoring                     # ← ③ 받은 scoring을 그대로 넘김
#     )
#     return scores, scores.mean(), scores.std()

In [8]:
# scoring만 교체
print("recall    :", evaluate_pipeline(X, y, task_model, task_cv, "recall")[1])
print("precision :", evaluate_pipeline(X, y, task_model, task_cv, "precision")[1])

# cv만 교체
print("TimeSeriesSplit F1:", evaluate_pipeline(X, y, task_model, TimeSeriesSplit(n_splits=5), "f1")[1])

# model만 교체
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
print("DecisionTree F1   :", evaluate_pipeline(X, y, dt, task_cv, "f1")[1])

recall    : 0.9105545248402391
precision : 0.063056377826583
TimeSeriesSplit F1: 0.10899165613165071
DecisionTree F1   : 0.8364047523584986


### Q5. 이번 Credit Card Fraud 데이터에서 선택한 CV와 그 이유를 작성하세요.

**답변:**  
StratifiedKFold. Credit Card 데이터는 소수 클래스인 사기 거래 비율이 0.001727로 매우 낮은 불균형 데이터이기 때문에 불균형 분류 문제에서는 StratifiedKFold을 사용한다.

### Q6. 전체 데이터에 StandardScaler를 먼저 fit하면 어떤 정보가 Validation Fold에서 Train 쪽으로 새어 들어갈 수 있나요?

**답변:**  
전체 데이터로 StandardScaler를 fit하면 Validation Fold에 들어갈 거래들의 평균과 표준편차가 새어 들어간다. 변환된 Train으로 학습하면서 평가 점수를 실제보다 좋게 볼 수 있다.

### Q7. 작성한 함수를 다른 모델·CV·평가지표에 재사용하려면 어떤 인자를 바꾸면 되나요?

**답변:**
evaluate_pipeline의 model, cv, scoring를 바꾼다. 
실제로 scoring만 바꾸면 recall 0.910555, precision 0.063056이 나오고,
cv만 TimeSeriesSplit으로 바꾸면 F1 평균이 0.108992,
model만 DecisionTree로 바꾸면 F1 평균이 0.836405가 된다.
다만 scoring 교체는 모델을 바꾸는 것이 아니라 같은 모델을 다른 지표로 보는 것이다.

### Q8. 재현성을 위해 어떤 설정을 고정했으며, Pipeline 생성 함수 안에서 전체 데이터로 전처리를 미리 학습하면 안 되는 이유는 무엇인가요?

**답변:**
random_state=42, shuffle=True로 섞는 순서와 모델의 무작위성을 고정했다.

각 Fold의 Train 데이터에서만 fit해야 하는데, 함수 안에서 미리 학습하면 Validation에 들어갈 거래까지 그 기준에 섞여 데이터 누수가 발생한다.

---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 교차검증을 사용하는 이유는 무엇인가요?
-> 한번의 데이터 분할 결과에만 의존하지 않고 여러번 학습하기 때문에 더 안정적인 추정치를 얻을 수 있다.

2. KFold와 StratifiedKFold의 가장 큰 차이는 무엇인가요?
-> KFold는 데이터를 K로 분할
-> StratifiedKFold는 각 Fold의 클래스 비율도 유지

3. 시계열 데이터에는 어떤 교차검증 방법이 적절한가요?
-> TimeSeriesSplit

4. StandardScaler는 전체 데이터가 아니라 어디에서 fit되어야 하나요?
-> Train 데이터에서만 fit해야함

5. SMOTE는 각 Fold의 Train / Validation 중 어디에만 적용되어야 하나요?
-> Train 데이터에서만 fit해야함

6. Pipeline이 데이터 누수 방지에 도움이 되는 이유는 무엇인가요?
-> 교차검증의 각 Fold마다 Train 데이터에서 전처리를 학습하고 Validation 데이터는 해당 기준을 적용하도록 과정을 묶어주기 때문 
